# Model training: Colab or Kaggle (GPU)

Everything that needs the GPU: zero-shot predictions, batch inference over the scenes, DINOv2 embeddings, fine-tuning RT-DETR and inference with the fine-tuned model. Every step runs with `uv run`.

**Data used (explicit):** `data/tiles_10cm/` = the 10 cm tiles and `data/tiles_10cm/labels/` = their GeoJSON annotations. The same folder is used by `fiftyone_workflow.ipynb`.

**Results to download** (step 8): `outputs/predictions/`, `outputs/batch/`, `data/embeddings/`. Put them in the same folders of your local repository.

## 1. Setup

The same start as `fiftyone_workflow.ipynb`.

In [ ]:
# Same start in both notebooks. On Colab or Kaggle this downloads the repository. On your computer
# the repository is already here and nothing is downloaded.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/emanuel-gf/FTL-Model-Training.git"   # set to your repository
REPO_DIR = "FTL-Model-Training"
IN_CLOUD = "COLAB_RELEASE_TAG" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
os.environ["MPLBACKEND"] = "Agg"

if not Path("pyproject.toml").exists():
    if not Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif IN_CLOUD and Path(".git").exists():
    subprocess.run(["git", "pull", "--ff-only"], check=False)
print("repository folder:", Path.cwd())

In [ ]:
# Same in both notebooks. Only GROUP differs: "gpu" in the cloud, "fo" on your computer.
from src.bootstrap import setup

GROUP = "gpu"
TILES_DIR = "data/tiles_10cm"       # EXPLICIT: the 10 cm tiles and their GeoJSON labels, used by both notebooks
TILES = setup(GROUP, TILES_DIR)

## 2. Download, tile and export the labels

Cloud only: the COGs and the RT-DETR checkpoint are downloaded here (they are not in the repository). The 10 cm tiles and the labels are rebuilt from the repository, so a new labelling pushed to git is picked up.

In [ ]:
if IN_CLOUD:
    !uv run python download_model.py --hub PekingU/rtdetr_r50vd --out models/rtdetr_r50

In [ ]:
RETILE = not (TILES / "tiles_index.csv").exists()          # True: tile the COGs now
if RETILE:
    if IN_CLOUD:
        !uv run python download_images.py --list list_of_ids.txt --out cog
    !uv run python tile_cogs.py --input cog --gsd 0.10 --tile 640 --overlap 128 --output {TILES_DIR}

!uv run python export_annotations.py --index {TILES_DIR}/tiles_index.csv --labels data/labelme/images --out {TILES_DIR}/labels --tile 640

## 3. Zero-shot predictions on the annotated tiles

`ZERO_SHOT_RUNS` lists the models. `ACTIVE` picks the ones to run now. Each run writes one JSON per annotated tile to `outputs/predictions/<name>/`.

In [ ]:
import json
import subprocess
from src.naming import run_name

ZERO_SHOT_RUNS = [
    dict(kind="sam2", weights="facebook/sam2.1-hiera-large", score_threshold=0.85, min_area_m2=10, max_area_m2=1500),
    dict(kind="owlv2", weights="google/owlv2-base-patch16-ensemble", score_threshold=0.2, queries=["a building"]),
    dict(kind="rtdetr", weights="PekingU/rtdetr_r50vd", score_threshold=0.3),
]
ACTIVE = [0, 1]                       # 0 = SAM2, 1 = OWLv2

for i in ACTIVE:
    options = dict(ZERO_SHOT_RUNS[i])
    kind, weights = options.pop("kind"), options.pop("weights")
    name = run_name(kind, weights)
    subprocess.run(["uv", "run", "python", "run_models.py", "--kind", kind, "--weights", weights,
                    "--name", name, "--mode", "tiles",
                    "--index", f"{TILES_DIR}/tiles_index.csv", "--labels", f"{TILES_DIR}/labels",
                    "--options", json.dumps(options)], check=True)

## 4. Batch over the scenes

One model over **every tile** of the chosen scenes, merged with NMS, one GeoJSON per scene in `outputs/batch/<name>/`. `BATCH_SCENES = None` runs all scenes.

In [ ]:
BATCH_RUN = 0                          # index in ZERO_SHOT_RUNS
BATCH_SCENES = None                    # None = all scenes, or e.g. ["68a4a953e50784296730bf0f"]

options = dict(ZERO_SHOT_RUNS[BATCH_RUN])
kind, weights = options.pop("kind"), options.pop("weights")
name = run_name(kind, weights)
cmd = ["uv", "run", "python", "run_models.py", "--kind", kind, "--weights", weights, "--name", name,
       "--mode", "scenes", "--index", f"{TILES_DIR}/tiles_index.csv", "--options", json.dumps(options)]
if BATCH_SCENES:
    cmd += ["--scenes", *BATCH_SCENES]
subprocess.run(cmd, check=True)

## 5. DINOv2 embeddings

One vector per tile, saved in one `.npz` file with the `tile_id` of each row inside it. For 5 cm tiles later, use `--tiles data/tiles_5cm` and an output in `data/embeddings/5cm/`.

In [ ]:
EMBED_OUT = "data/embeddings/10cm/dinov2_base.npz"
subprocess.run(["uv", "run", "python", "embed.py", "--tiles", TILES_DIR,
                "--model", "facebook/dinov2-base", "--out", EMBED_OUT], check=True)

## 6. Fine-tune RT-DETR

Trains on the same tiles and labels (`data.tiles_dir` is set explicitly). The best checkpoint is exported to `outputs/<run>/hf_model/`.

In [ ]:
EPOCHS = 10
subprocess.run(["uv", "run", "python", "train.py",
                f"train.epochs={EPOCHS}", "train.batch_size=4",
                f"data.tiles_dir={TILES_DIR}", "model.checkpoint=models/rtdetr_r50"], check=True)

## 7. Inference over the COGs with the fine-tuned model

Uses the last exported model. Detections go to `outputs/geojson/`.

In [ ]:
from pathlib import Path
exported = sorted(Path("outputs").glob("*/hf_model"))
MODEL_CHECKPOINT = str(exported[-1]) if exported else "models/rtdetr_r50"
print("model:", MODEL_CHECKPOINT)
subprocess.run(["uv", "run", "python", "infer.py", f"model.checkpoint={MODEL_CHECKPOINT}",
                "inference.score_threshold=0.4", "inference.gsd=0.10"], check=True)

## 8. Results for your computer

Packs the predictions, the batch GeoJSON and the embeddings into one zip. Unzip it at the root of your local repository: the paths inside keep their folders.

In [ ]:
import zipfile
from pathlib import Path

with zipfile.ZipFile("results_for_local.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["outputs/predictions", "outputs/batch", "data/embeddings"]:
        for f in Path(folder).rglob("*"):
            if f.is_file():
                z.write(f, f.as_posix())
print("written: results_for_local.zip")